In [4]:
from openpyxl import load_workbook
import openpyxl
import shutil

# "住民台帳"シートから住民情報リスト(j_info)と住民氏名リスト(j_name)を作成して出力する関数
def get_j_info(ws):
    # 各住民の情報を取得して仮に入力しておくリスト
    tmp_list = []
    # 各住民の住民台帳の情報
    j_info = []
    # 住民の氏名
    j_name = []
    for row in range(2, ws.max_row + 1):
        # 住民の氏名を追記(row行目のA列が氏名)
        j_name.append(ws.cell(row=row, column=1).value)
        # 1列目から最終列まで列ループ
        for clm in range(1, ws.max_column + 1):
            tmp_list.append(ws.cell(row=row, column=clm).value)
        # 1行分(1人分)のリストを j_info に追加 → 2次元リストになる
        j_info.append(tmp_list)
        tmp_list = []
    return j_info, j_name


class Jumin_all_info:
    def __init__(self, jumin, ws):
        self.jumin = jumin
        self.ws = ws

    # 各住民の引数で指定した光熱費の情報(掲載月すべてを含む)を取得する関数
    def get_jumin_bill_info(self):
        result = []
        tmp_list = []
        # ヘッダーを除いた2行目から行ループ開始
        for row in range(2, self.ws.max_row + 1):
            # セルの中身がjuminと一致した場合(氏名の検索)
            if self.ws.cell(row=row, column=1).value == self.jumin:
                # 1列目から最終列まで列ループ開始
                for clm in range(1, self.ws.max_column + 1):
                    tmp_list.append(self.ws.cell(row=row, column=clm).value)
                result.append(tmp_list)
                tmp_list = []
        return result



def main():
    path = "./Database.xlsx"
    wb = load_workbook(path, data_only=True)
    names = ["住民台帳", "水道料金", "電気料金", "ガス料金"]

    sheets = []
    for name in names:
        sheets.append(wb[name])
    # "住民台帳", "水道料金", "電気料金", "ガス料金"シートを表す
    ws_jumin, ws_water, ws_denki, ws_gas = sheets
    # j_info:各住民の住民台帳の情報
    # j_name:各住民の氏名
    j_info, j_name = get_j_info(ws_jumin)

    ### 全住民の全光熱費情報を最初から順番に格納する ((住民台帳は含まない) 3次元リスト) ###
    j_bill_info = []
    # 住民の数だけfor文でループする
    for jcnt in range(len(j_name)):
        # 一人分の全光熱費情報(全月)を格納する一時リスト
        person = []
        # "水道料金", "電気料金", "ガス料金"をfor文でループする
        for i in range(1, len(names)):
            # このループで一人ずつ"水道料金"→"電気料金"→"ガス料金"情報を取得する
            info_obj = Jumin_all_info(j_name[jcnt], wb[names[i]])
            # person[0]:1人目の水道料金
            person.append(info_obj.get_jumin_bill_info())
        # 一人分の全情報が取得できたのでj_bill_infoリストへ追加する
        j_bill_info.append(person)
# j_bill_info                 ← 1段目：住民の一覧
# ├─ [0] 佐藤さん           ← 2段目：料金の種類
# │    ├─ [0] 水道         ← 3段目：明細の行
# │    │    ├─ [0] ['佐藤　滋', 1134001, 1, 30, 400, 12000]   1月
# │    │    ├─ [1] ['佐藤　滋', 1134001, 2, 33, 400, 13200]   2月
# │    │    ├─ [2] ['佐藤　滋', 1134001, 3, 55, 400, 22000]   3月
# │    │    └─ [3] ['佐藤　滋', 1134001, 4, 46, 400, 18400]   4月
# │    ├─ [1] 電気（3行）
# │    └─ [2] ガス（5行）
# ├─ [1] 鈴木さん
# │    ├─ [0] 水道
# │    ├─ [1] 電気
# │    └─ [2] ガス
# └─ …

    # Database.xlsxの各シートのヘッダー部以外をすべて削除する→Database_delete.xlsx
    shutil.copy(path, "Database_delete.xlsx")
    wb_tmp = load_workbook("Database_delete.xlsx")
    for ws in wb_tmp:
        if ws.max_row >= 2:
            ws.delete_rows(2, ws.max_row - 1)
    wb_tmp.save("Database_delete.xlsx")

    # 各住民のファイルを個別に作成・処理するためのループ
    for idx, name in enumerate(j_name):
        file_path = f"{name}.xlsx"
        shutil.copy("Database_delete.xlsx", file_path)

        # 新しく作ったファイル(ワークブック)をロード
        wb_cur = load_workbook(file_path)

        ### "住民台帳"シートに必要情報を書き込み ###
        ws_cur = wb_cur["住民台帳"]
        for col, val in enumerate(j_info[idx], start=1):
            ws_cur.cell(row=2, column=col).value = val

        ### "水道料金"シートに必要情報を書き込み ###
        ws_cur = wb_cur["水道料金"]
        water_data = j_bill_info[idx][0]
        for row_offset, record in enumerate(water_data, start=2):
            for col_idx, val in enumerate(record, start=1):
                ws_cur.cell(row=row_offset, column=col_idx).value = val
        # A,B列の3行目以降を削除
        for row in range(3, ws_cur.max_row + 1):
            for col in range(1, 3):
                ws_cur.cell(row=row, column=col).value = None

        ### "電気料金"シートに必要情報を書き込み ###
        ws_cur = wb_cur["電気料金"]
        electricity_data = j_bill_info[idx][1]
        for row_offset, record in enumerate(electricity_data, start=2):
            for col_idx, val in enumerate(record, start=1):
                ws_cur.cell(row=row_offset, column=col_idx).value = val
        # A,B列の3行目以降を削除
        for row in range(3, ws_cur.max_row + 1):
            for col in range(1, 3):
                ws_cur.cell(row=row, column=col).value = None

        ### "ガス料金"シートに必要情報を書き込み ###
        ws_cur = wb_cur["ガス料金"]
        gas_data = j_bill_info[idx][2]
        for row_offset, record in enumerate(gas_data, start=2):
            for col_idx, val in enumerate(record, start=1):
                ws_cur.cell(row=row_offset, column=col_idx).value = val
        # A,B列の3行目以降を削除
        for row in range(3, ws_cur.max_row + 1):
            for col in range(1, 3):
                ws_cur.cell(row=row, column=col).value = None

        # 変更された個別ブックを保存します
        wb_cur.save(file_path)
        print(f"{file_path} の作成と書き込みが完了しました。")

# このファイルを直接実行したときだけ main() を呼び出す、というお決まりの書き方
if __name__ == "__main__":
    main()

佐藤　滋.xlsx の作成と書き込みが完了しました。
鈴木　健一.xlsx の作成と書き込みが完了しました。
田中　芳子.xlsx の作成と書き込みが完了しました。
田上　良太.xlsx の作成と書き込みが完了しました。
竹本　佳代子.xlsx の作成と書き込みが完了しました。
大元　久.xlsx の作成と書き込みが完了しました。
